In [1]:
spark.sql("CREATE OR REPLACE TABLE gold_dim_agent    AS SELECT AgentID,AgentName,OfficeCity,AgentType,Designation,SpecializationType,CommissionPercent,AgentRating,AgentStatus FROM silver_agent")
spark.sql("CREATE OR REPLACE TABLE gold_dim_customer AS SELECT CustomerID,Customer_Name,CustomerType,City,State,Occupation,IncomeBracket,CustomerStatus FROM silver_customer")
spark.sql("CREATE OR REPLACE TABLE gold_dim_builder  AS SELECT BuilderID,BuilderName,BuilderType,HeadOfficeCity,BuilderRating,RERARegisteredFlag FROM silver_builder")
spark.sql("CREATE OR REPLACE TABLE gold_dim_property AS SELECT PropertyID,PropertyType,PropertyCategory,BuilderID,City,State,BuiltUpAreaSqFt,TotalPropertyValue,AvailabilityStatus FROM silver_property")

spark.sql("""CREATE OR REPLACE TABLE gold_dim_date AS
 SELECT d AS DateKey,year(d) Year,month(d) Month,date_format(d,'MMMM') MonthName,quarter(d) Quarter
 FROM (SELECT explode(sequence(to_date('2019-01-01'),to_date('2027-12-31'),interval 1 day)) d)""")

spark.sql("""CREATE OR REPLACE TABLE gold_fact_sales AS
 SELECT SalesID,SaleDate AS DateKey,PropertyID,AgentID,CustomerID,FinalSalePrice,TotalDealValue,
        CommissionAmount,GSTAmount,SaleType,PaymentType,SaleStatus FROM silver_sales""")
spark.sql("""CREATE OR REPLACE TABLE gold_fact_lease AS
 SELECT LeaseID,AgreementDate AS DateKey,PropertyID,AgentID,CustomerID,MonthlyRent,SecurityDeposit,
        BrokerageAmount,LeaseTenureMonths,LeaseType,LeaseStatus FROM silver_lease""")

spark.sql("""CREATE OR REPLACE TABLE gold_kpi_agent_performance AS
 SELECT a.AgentID,a.AgentName,COUNT(s.SalesID) AS Deals,
        SUM(s.TotalDealValue) AS TotalValue,
        ROUND(AVG(s.TotalDealValue),0) AS AvgDeal
 FROM silver_agent a LEFT JOIN silver_sales s ON a.AgentID=s.AgentID
 GROUP BY a.AgentID,a.AgentName""")
spark.sql("""CREATE OR REPLACE TABLE gold_kpi_monthly_sales AS
 SELECT year(SaleDate) Year,month(SaleDate) Month,COUNT(*) Deals,SUM(TotalDealValue) Revenue
 FROM silver_sales GROUP BY year(SaleDate),month(SaleDate) ORDER BY Year,Month""")
print("gold built")

StatementMeta(, 2a1f58c5-7351-4096-a2c6-0e01f99fae89, 3, Finished, Available, Finished, False)

gold built


In [1]:
spark.sql("""CREATE OR REPLACE TABLE gold_dim_agent AS
 SELECT AgentID,AgentName,OfficeCity,AgentType,Designation,SpecializationType,
        CommissionPercent,AgentRating,AgentStatus,ReportingManagerID FROM silver_agent""")
spark.sql("""CREATE OR REPLACE TABLE gold_dim_property AS
 SELECT PropertyID,PropertyType,PropertyCategory,BuilderID,AgentID,OwnerID,
        City,State,BuiltUpAreaSqFt,TotalPropertyValue,AvailabilityStatus FROM silver_property""")
spark.sql("""CREATE OR REPLACE TABLE gold_dim_customer AS
 SELECT CustomerID,Customer_Name,CustomerType,City,State,Occupation,
        IncomeBracket,AssignedAgentID,CustomerStatus FROM silver_customer""")
print("FK columns ready")

StatementMeta(, 17cafc7b-06c0-411d-bd69-bdfb28840501, 3, Finished, Available, Finished, False)

FK columns ready


In [1]:
# read the loose delta data and re-save as proper registered tables in Gold_LH
gold_tables = ["gold_dim_agent","gold_dim_builder","gold_dim_customer","gold_dim_date",
               "gold_dim_property","gold_fact_sales","gold_fact_lease",
               "gold_kpi_agent_performance","gold_kpi_monthly_sales"]

base = "abfss://<WORKSPACE>@onelake.dfs.fabric.microsoft.com/Gold_LH.Lakehouse/Tables"

for t in gold_tables:
    try:
        df = spark.read.format("delta").load(f"{base}/{t}")
        df.write.format("delta").mode("overwrite").saveAsTable(t)   # registers properly
        print("re-registered", t)
    except Exception as e:
        print("skip", t, str(e)[:80])

StatementMeta(, ead1ad41-cac9-490f-a77e-26ab5da79780, 3, Finished, Available, Finished, False)

skip gold_dim_agent Illegal character in authority at index 8: abfss://<WORKSPACE>@onelake.dfs.fabri
skip gold_dim_builder Illegal character in authority at index 8: abfss://<WORKSPACE>@onelake.dfs.fabri
skip gold_dim_customer Illegal character in authority at index 8: abfss://<WORKSPACE>@onelake.dfs.fabri
skip gold_dim_date Illegal character in authority at index 8: abfss://<WORKSPACE>@onelake.dfs.fabri
skip gold_dim_property Illegal character in authority at index 8: abfss://<WORKSPACE>@onelake.dfs.fabri
skip gold_fact_sales Illegal character in authority at index 8: abfss://<WORKSPACE>@onelake.dfs.fabri
skip gold_fact_lease Illegal character in authority at index 8: abfss://<WORKSPACE>@onelake.dfs.fabri
skip gold_kpi_agent_performance Illegal character in authority at index 8: abfss://<WORKSPACE>@onelake.dfs.fabri
skip gold_kpi_monthly_sales Illegal character in authority at index 8: abfss://<WORKSPACE>@onelake.dfs.fabri
